# Baseline — Qwen2.5-VL-3B (zero-shot) on ScreenSpot-V2

Smoke test for the shared infrastructure. Run this once before any task notebook.

**Hypothesis to confirm:** unadapted Qwen2.5-VL-3B is around 27% on ScreenSpot-V2 (per the proposal, §3). If this notebook reports something within ±5 pp of that, the eval harness is wired correctly.

In [ ]:
# Environment check. Locally, `uv sync` in the repo root has already installed ais5.
# On Colab / Kaggle, clone your repo first, then run `uv sync --extra notebook`
# (or `pip install -e .`) before re-running this cell.
try:
    import ais5
    print(f"ais5 v{ais5.__version__} ready")
except ImportError:
    print("ais5 is not installed in this kernel.")
    print("  Local: from the repo root, run `uv sync --extra notebook`")
    print("  Colab: clone your repo, cd into it, then run the same command.")
    raise

In [2]:
from ais5.data import load_benchmark
from ais5.eval import evaluate_model, by_target_size, by_ui_type
from ais5.models import get_model
from ais5.utils import set_global_seed, setup_logging

setup_logging()
set_global_seed(42)

In [ ]:
model = get_model('qwen2.5-vl-3b')
samples = list(load_benchmark('screenspot-v2'))
print(f'{len(samples)} samples loaded')
print(f'first sample: {samples[0]}')

[05/06/26 14:20:37] INFO     HTTP Request: HEAD                                                                    
                             https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/config.json "HTTP/1.1 
                             307 Temporary Redirect"

                    WARNING  Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN
                             to enable higher rate limits and faster downloads.

                    INFO     HTTP Request: HEAD                                                                    
                             https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-3B-Instruct/66285546d2
                             b821cf421d4f5eb2576359d3770cd3/config.json "HTTP/1.1 200 OK"

                    INFO     HTTP Request: GET                                                                     
                             https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-3B-Instruct/66285546d2
                             b821cf421d4f5eb2576359d3770cd3/config.json "HTTP/1.1 200 OK"

config.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[05/06/26 14:20:39] INFO     HTTP Request: HEAD                                                                    
                             https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/66285546d2b821cf421d4f5eb25
                             76359d3770cd3/model-00001-of-00002.safetensors "HTTP/1.1 302 Found"

                    INFO     HTTP Request: HEAD                                                                    
                             https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/66285546d2b821cf421d4f5eb25
                             76359d3770cd3/model-00002-of-00002.safetensors "HTTP/1.1 302 Found"

                    INFO     HTTP Request: GET                                                                     
                             https://huggingface.co/api/models/Qwen/Qwen2.5-VL-3B-Instruct/xet-read-token/66285546d
                             2b821cf421d4f5eb2576359d3770cd3 "HTTP/1.1 200 OK"

                    INFO     HTTP Request: GET                                                                     
                             https://huggingface.co/api/models/Qwen/Qwen2.5-VL-3B-Instruct/xet-read-token/66285546d
                             2b821cf421d4f5eb2576359d3770cd3 "HTTP/1.1 200 OK"

In [ ]:
# Limit to 50 for a quick sanity run; remove `limit=` for the full eval.
run = evaluate_model(model, samples, benchmark='screenspot-v2', limit=50)
print(f'accuracy = {run.accuracy:.4f} (n={len(run.results)})')
print(f'avg latency = {run.avg_latency_ms:.1f} ms' if run.avg_latency_ms else '')

In [ ]:
by_target_size(run.results)

In [ ]:
by_ui_type(run.results)

## Sanity checks

- Did the parser succeed on most samples? Look for `<click>` patterns in `run.results[i].raw_response`.
- Are the wrong-answer points falling near the bbox or far away? Far → prompt issue. Near → resolution issue.
- Is the average latency reasonable (~1-3 s on Colab T4)? Much higher → check `device_map`.